In [ ]:
# IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# INITIALIZE JS FOR FORCE PLOTS
# Useful for Jupyter Notebook / JupyterLab
shap.initjs()

# MULTICLASS CLASSIFICATION

In [ ]:
iris = load_iris()

In [ ]:
X_multi = pd.DataFrame(iris.data,columns=iris.feature_names)

y_multi = pd.Series(iris.target,name="target")

In [ ]:
print("\nMulticlass dataset:")
print(X_multi.shape)

In [ ]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = (train_test_split(X_multi,y_multi,test_size=0.20,random_state=4))

In [ ]:
multi_model = RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1)

In [ ]:
multi_model.fit(X_train_multi,y_train_multi)

In [ ]:
multi_predictions = multi_model.predict(X_test_multi)

In [ ]:
print("\nMulticlass Accuracy:")

print(accuracy_score(y_test_multi,multi_predictions))

print("\nClassification Report:")

print(classification_report(y_test_multi,multi_predictions,target_names=iris.target_names)

## CREATE MULTICLASS SHAP EXPLAINER

In [ ]:
multi_explainer = shap.TreeExplainer(multi_model)

## CALCULATE MULTICLASS SHAP VALUES

In [ ]:
multi_shap_values = multi_explainer(X_test_multi)

print("\nMulticlass SHAP shape:")
print(multi_shap_values.values.shape)

## MULTICLASS SHAP STRUCTURE


 Modern SHAP commonly returns:

     samples × features × classes

 Example:

     30 × 4 × 3

 means:

     30 samples
     4 features
     3 classes

In [ ]:
multi_values = multi_shap_values.values

if multi_values.ndim != 3:

    raise ValueError(
        "Expected multiclass SHAP values to have "
        "3 dimensions: samples × features × classes. "
        f"Received: {multi_values.shape}")


n_samples = multi_values.shape[0]
n_features = multi_values.shape[1]
n_classes = multi_values.shape[2]

print("\nNumber of samples:", n_samples)
print("Number of features:", n_features)
print("Number of classes:", n_classes)

## CREATE CLASS-SPECIFIC SHAP EXPLANATION

In [ ]:
def get_multiclass_class_explanation(shap_explanation,class_id,X_data):
    """
    Extract SHAP explanation for one class.

    Input:
        SHAP Explanation with:
        samples × features × classes

    Output:
        samples × features
    """

    values = shap_explanation.values[:,:,class_id]

    base_values = shap_explanation.base_values

    # Base values may be:
    #
    # samples × classes
    #
    # or possibly a single value.

    if np.ndim(base_values) == 2:
        selected_base_values = (base_values[:, class_id])

    elif np.ndim(base_values) == 1:
        selected_base_values = base_values

    else:
        selected_base_values = base_values

    return shap.Explanation(
        values=values,
        base_values=selected_base_values,
        data=X_data.values,
        feature_names=X_data.columns.tolist()
    )

## CHOOSE CLASS TO EXPLAIN

In [ ]:
selected_multiclass_class = 1

selected_class_name = iris.target_names[selected_multiclass_class]

print("\nSelected multiclass class:")

print(selected_multiclass_class,"->",selected_class_name)

## EXTRACT SELECTED CLASS SHAP VALUES

In [ ]:
class_1_shap = get_multiclass_class_explanation(
    multi_shap_values,
    selected_multiclass_class,
    X_test_multi)

## BAR PLOT

In [ ]:
shap.plots.bar(
    class_1_shap,
    max_display=10,
    show=False)

plt.title(f"SHAP Feature Importance - Class: {selected_class_name}")

plt.tight_layout()
plt.show()

## BEESWARM

In [ ]:
shap.plots.beeswarm(
    class_1_shap,
    max_display=10,
    show=False)

plt.title(f"SHAP Beeswarm - Class: {selected_class_name}")

plt.tight_layout()
plt.show()

## WATERFALL

In [ ]:
multi_sample_index = 0

shap.plots.waterfall(
    class_1_shap[multi_sample_index],
    max_display=10,
    show=False)

plt.title(f"SHAP Waterfall - Class: {selected_class_name}")

plt.tight_layout()
plt.show()

## FORCE PLOT

In [ ]:
shap.plots.force(class_1_shap[multi_sample_index])

## DECISION PLOT

In [ ]:
shap.plots.decision(
    class_1_shap.base_values[
        multi_sample_index
    ],
    class_1_shap.values[
        multi_sample_index
    ],
    feature_names=X_test_multi.columns.tolist(),
    show=False)

plt.title(f"SHAP Decision Plot - Class: {selected_class_name}")

plt.tight_layout()
plt.show()

## DEPENDENCE / SCATTER
 Determine most important feature for selected class.


In [ ]:
multi_mean_abs_shap = np.abs(
    class_1_shap.values
).mean(axis=0)

multi_best_feature_index = np.argmax(
    multi_mean_abs_shap)

multi_best_feature = (
    X_test_multi.columns[
        multi_best_feature_index
    ]
)

In [ ]:
shap.plots.scatter(
    class_1_shap[
        :,
        multi_best_feature
    ],
    show=False)

plt.title(f"SHAP Dependence - "f"{multi_best_feature} "f"for {selected_class_name}")

plt.tight_layout()
plt.show()

# EXPLAIN ALL MULTICLASS OUTPUTS

## CLASS-SPECIFIC SHAP EXPLANATION FOR EVERY CLASS

In [ ]:
for class_id in range(n_classes):

    class_name = iris.target_names[class_id]

    print("CLASS:",class_id,"->",class_name)
    
    class_explanation = (
        get_multiclass_class_explanation(
            multi_shap_values,
            class_id,
            X_test_multi))
    
    # Bar plot
    shap.plots.bar(
        class_explanation,
        max_display=10,
        show=False)

    plt.title(f"SHAP Global Importance - {class_name}")

    plt.tight_layout()
    plt.show()

    # Beeswarm
    shap.plots.beeswarm(
        class_explanation,
        max_display=10,
        show=False)

    plt.title(f"SHAP Beeswarm - {class_name}")

    plt.tight_layout()
    plt.show()

# EXPLAIN THE MODEL'S PREDICTED CLASS FOR ONE SAMPLE

## SELECT A MULTICLASS SAMPLE

In [ ]:
sample = X_test_multi.iloc[[multi_sample_index]]

actual_class = y_test_multi.iloc[multi_sample_index]

predicted_class = multi_model.predict(sample)[0]

probabilities = multi_model.predict_proba(sample)[0]

In [ ]:
print("MULTICLASS INDIVIDUAL PREDICTION")

print("\nActual class:")
print(actual_class,"->",iris.target_names[actual_class])

print("\nPredicted class:")
print(predicted_class,"->",iris.target_names[predicted_class])

print("\nClass probabilities:")
for class_id, probability in enumerate(probabilities):
    print(iris.target_names[class_id],":",probability)

## EXPLAIN THE PREDICTED CLASS

In [ ]:
predicted_class_shap = (
    get_multiclass_class_explanation(
        multi_shap_values,
        predicted_class,
        X_test_multi))

In [ ]:
## WATERFALL FOR PREDICTED CLASS

In [ ]:
shap.plots.waterfall(
    predicted_class_shap[
        multi_sample_index
    ],
    max_display=10,
    show=False)

plt.title("Why the Model Predicted: "+ iris.target_names[predicted_class])

plt.tight_layout()
plt.show()

## FORCE PLOT FOR PREDICTED CLASS

In [ ]:
shap.plots.force(
    predicted_class_shap[
        multi_sample_index])

## DECISION PLOT FOR PREDICTED CLASS

In [ ]:
shap.plots.decision(
    predicted_class_shap.base_values[
        multi_sample_index
    ],
    predicted_class_shap.values[
        multi_sample_index
    ],
    feature_names=X_test_multi.columns.tolist(),
    show=False)

plt.title("Decision Plot - Predicted Class: "+ iris.target_names[predicted_class])

plt.tight_layout()
plt.show()